# Force Model/Constellation Testing

Test the use of force model with the Constellation class. 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

# import constellation clase
from contigo.constellation import Constellation
from contigo.forces.third_body_acc import ThirdBody
from contigo.forces.grav_pot import EarthPotential
from contigo.forces.srp_acc import SRPAcc

In [3]:
sw_e = pd.read_hdf("./data/ESA_pod.hdf")
sw_o = pd.read_hdf("./data/ore_d.hdf")

print(sw_e.columns)
print(sw_o.columns)

Index(['index', 'sat', 'x', 'y', 'z', 'DateTime', 'vx', 'vy', 'vz',
       'EstSat.EarthFixed.X', 'EstSat.EarthFixed.Y', 'EstSat.EarthFixed.Z',
       'EstSat.EarthFixed.VX', 'EstSat.EarthFixed.VY', 'EstSat.EarthFixed.VZ',
       'EstSat.TAIGregorian', 'eci_x', 'eci_y', 'eci_z', 'eci_vx', 'eci_vy',
       'eci_vz', 'eg_x', 'eg_y', 'eg_z', 'sg_x', 'sg_y', 'sg_z', 'mg_x',
       'mg_y', 'mg_z', 'srp_x', 'srp_y', 'srp_z'],
      dtype='str')
Index(['eci_x', 'eci_y', 'eci_z', 'eci_vx', 'eci_vy', 'eci_vz', 'eci_sn_ax',
       'eci_sn_ay', 'eci_sn_az', 'eci_mn_ax', 'eci_mn_ay', 'eci_mn_az',
       'ecef_sn_ax', 'ecef_sn_ay', 'ecef_sn_az', 'ecef_mn_ax', 'ecef_mn_ay',
       'ecef_mn_az', 'ecef_sn_px', 'ecef_sn_py', 'ecef_sn_pz', 'ecef_mn_px',
       'ecef_mn_py', 'ecef_mn_pz', 'earth_gp', 'DateTime', 'ecef_x', 'ecef_y',
       'ecef_z', 'ecef_vx,', 'ecef_vy', 'ecef_vz', 'edr', 'denom'],
      dtype='str')


### Start with the Third Body Accelerations

In [4]:
# create a Constellation object from the ESA POD file
# and calculate thirdbody acceleration from ThirdBody
hdf_sc = Constellation(state_file=r'D:\GitHub\contigo_edr\data\ESA_pod.hdf', 
                    time_col='DateTime', x_col='x', y_col='y', z_col='z',
                    vx_col='vx', vy_col='vy', vz_col='vz', 
                    sc_id_col='filename', sc_fn_slc=slice(-11,-8),
                    tscale_input='GPS', 
                    sc_mass=4.3e+02, cr=1.8, srp_area=1)

# Chain Initialization and acceleration derivation together
acc = ThirdBody(body=['SUN','MOON']).acceleration(hdf_sc)

# Compare to the orekit derived accelerations
acc_s = acc['ESA'][0,:,:]
acc_m = acc['ESA'][1,:,:]

# orekit accelerations where in m/s
ore_s = sw_o[['ecef_sn_ax', 'ecef_sn_ay', 'ecef_sn_az']].to_numpy()/1000.
ore_m = sw_o[['ecef_mn_ax', 'ecef_mn_ay', 'ecef_mn_az']].to_numpy()/1000.             

print(np.allclose(acc_s,ore_s))
print(np.allclose(acc_s,ore_m))



INFO:Loading Kernel - D:\GitHub\contigo_edr\contigo\data\de440s.bsp
INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\naif0012.tls
INFO:Loading Kernel - D:\GitHub\contigo_edr\contigo\data\earth_latest_high_prec.bpc


True
True


In [5]:
acc

{'ESA': array([[[-9.00593868e-11,  2.80000065e-10,  9.67684671e-12],
         [-9.29426873e-11,  2.79420227e-10,  1.15478522e-11],
         [-9.58155164e-11,  2.78810575e-10,  1.34174444e-11],
         ...,
         [-2.65443983e-10, -9.03881075e-12,  1.90759902e-10],
         [-2.63011733e-10, -1.15689215e-11,  1.92118514e-10],
         [-2.60551295e-10, -1.41011777e-11,  1.93453780e-10]],
 
        [[-1.81861563e-10,  7.29546172e-10,  5.87371096e-11],
         [-1.88527722e-10,  7.28097078e-10,  6.42529800e-11],
         [-1.95173071e-10,  7.26569792e-10,  6.97579894e-11],
         ...,
         [-3.99667989e-10, -8.00838078e-11,  3.95561683e-10],
         [-3.92872928e-10, -8.30031643e-11,  3.98325732e-10],
         [-3.86035372e-10, -8.59190698e-11,  4.01039822e-10]]],
       shape=(2, 138240, 3))}

### Earth Potential

In [6]:
geo_pot = EarthPotential().potential(hdf_sc)
print(np.allclose(geo_pot['ESA'],sw_o['earth_gp'].to_numpy()/1000**2))
print(geo_pot['ESA'][0:5])
print(sw_o['earth_gp'].to_numpy()[0:5]/1000**2)

INFO:Loading potential file D:\GitHub\contigo_edr\contigo\data\EIGEN-2.gfc


True
[57.9093057  57.90821267 57.90711047 57.9059993  57.90487937]
[57.90930572 57.90821267 57.90711046 57.90599927 57.90487933]


### SRP Accelerations

In [8]:
srp_acc = SRPAcc(apistartup="api_startup_file.txt",
            gmat_install="C:/Users/murph/GMAT_R2025a/").acceleration(hdf_sc, None)

srp_test = np.linalg.norm(srp_acc['ESA'],axis=1)
gmat_acc = np.linalg.norm(sw_e[['srp_x', 'srp_y', 'srp_z']].to_numpy(),axis=1)

print(np.allclose(srp_test,gmat_acc))
print(srp_test[0:4])
print(gmat_acc[0:4])

INFO:Setting up GMAT API.


True
[1.97421528e-11 1.97421596e-11 1.97421665e-11 1.97421733e-11]
[1.97421528e-11 1.97421596e-11 1.97421665e-11 1.97421733e-11]
